# 13 - C2 candidate recall (Gate A)

Before any ranker: is the truth **retrievable** from the universe at a given mass tolerance?

* `external_only` reachability (THE Gate-A number): the truth counts only if a non-TRAIN source (COCONUT / PubChem) contains its connectivity -- what a hidden Class-2 molecule experiences.
* `universe` reachability: truth anywhere in the universe (TRAIN included) -- ~100% by construction for dev queries; isolates mass accuracy + pool density.

Candidate order for Recall@k = retrieval order (abs ppm, then candidate_id). Fold views exclude that fold's C3 truths. `pool_share_with_reference` = share of pool candidates that still own reference spectra after hiding (if ~1.0, C2 truths would be the odd ones out -> shortcut risk).

If external coverage is poor: **fix the candidate universe first** -- do not compensate with a ranker. CPU only.

In [ ]:
import os, sys, json
from pathlib import Path
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
DRIVE_ROOT = Path(os.environ.setdefault('ENVEDA_DRIVE_ROOT', '/content/drive/MyDrive/EnvedaCASMI'))
REPO_ROOT = Path(os.environ.setdefault('ENVEDA_REPO_ROOT', str(DRIVE_ROOT / 'repo')))
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
from casmi.workspace.config import load_v2_config, input_path
from casmi.workspace.environment import set_seeds, system_report
CFG, P = load_v2_config(REPO_ROOT / 'configs' / 'casmi_v2_colab.yaml')
P.ensure()
SEED = set_seeds(CFG['runtime']['seed'])
NOTEBOOK = '13_colab_candidate_recall'
print(system_report())

In [ ]:
import numpy as np, pandas as pd
from casmi.workspace.artifacts import read_metadata
from casmi.candidates.mass_index import CandidateMassIndex
from casmi.candidates.universe import load_candidate_keys, keys_to_ids, read_candidates
UNIVERSE_ROOT = P.candidate_db_dir / 'universe'
assert (UNIVERSE_ROOT / 'universe_manifest.json').exists(), 'run notebook 12 first'
REG = pd.read_parquet(P.processed_dir / 'validation_regimes.parquet')
REG_META = read_metadata(P.processed_dir, 'validation_regimes.metadata.json')
print('regime table c2_mode:', REG_META['c2_mode'])
if REG_META['c2_mode'] != 'external_universe':
    print('WARNING: PRELIMINARY regimes (built before the universe existed) -- re-run notebook 11 with REBUILD = True')
dev = pd.read_parquet(input_path(CFG, P, 'dev_queries'), columns=['query_id', 'neutral_mass', 'ionization_mode', 'adduct', 'instrument_type'])
dev['query_id'] = dev['query_id'].astype(str)
Q = REG.dropna(subset=['regime']).merge(dev, on='query_id', how='left', validate='one_to_one')
IDX = CandidateMassIndex.load(UNIVERSE_ROOT / 'index')
UKEYS = load_candidate_keys(UNIVERSE_ROOT)
Q['truth_id'] = keys_to_ids(UKEYS, Q['true_connectivity_key'])
TR = read_candidates(UNIVERSE_ROOT, Q.loc[Q.truth_id >= 0, 'truth_id'].unique(), columns=['candidate_sources', 'molecular_formula'])
TR = TR.set_index('candidate_id')
print(f'queries: {len(Q):,} | universe candidates: {IDX.n_candidates:,} | truths found in universe: {(Q.truth_id >= 0).mean():.4f}')

In [ ]:
from casmi.candidates.recall import formula_element_class, source_label, truth_reachable
tr = TR.reindex(Q['truth_id']).reset_index(drop=True)
tr['candidate_id'] = Q['truth_id'].to_numpy()
Q['reach_universe'] = truth_reachable(tr.assign(candidate_sources=tr['candidate_sources'].apply(lambda s: s if isinstance(s, (list, np.ndarray)) else [])), 'universe')
Q['reach_external'] = truth_reachable(tr.assign(candidate_sources=tr['candidate_sources'].apply(lambda s: s if isinstance(s, (list, np.ndarray)) else [])), 'external_only')
Q['formula_class'] = tr['molecular_formula'].map(formula_element_class).to_numpy()
Q['truth_sources'] = tr['candidate_sources'].apply(lambda s: source_label(s if isinstance(s, (list, np.ndarray)) else [], drop_train=True)).to_numpy()
display(Q.groupby('regime')[['reach_universe', 'reach_external']].mean())

In [ ]:
# candidates that own reference spectra (TRAIN with spectra) -- metadata for the shortcut diagnostic only
off = json.loads((UNIVERSE_ROOT / 'bucket_offsets.json').read_text())['offsets']
HAS_REF = np.zeros(IDX.n_candidates, dtype=bool)
for b, start in off.items():
    t = pd.read_parquet(UNIVERSE_ROOT / 'buckets' / f'bucket={b}.parquet', columns=['has_reference_spectrum'])
    HAS_REF[start:start + len(t)] = t['has_reference_spectrum'].fillna(False).to_numpy(bool)
print(f'candidates with reference spectra: {HAS_REF.mean():.4f}')

In [ ]:
# recall sweep per fold view (C3 truths of the fold excluded; hidden truths lose their reference flag)
from casmi.candidates.recall import recall_sweep
from casmi.validation.regimes import hidden_sets, removed_candidate_mask
PPM = CFG['recall']['ppm_grid']
SWEEPS = []
for f in sorted(Q['fold'].unique()):
    hs = hidden_sets(REG, f)
    excl = removed_candidate_mask(UKEYS, hs['removed_structure_keys'])
    has_ref_f = HAS_REF.copy()
    hid = keys_to_ids(UKEYS, sorted(hs['hidden_reference_keys']))
    has_ref_f[hid[hid >= 0]] = False
    qf = Q[(Q['fold'] == f) & Q['regime'].isin(['C1', 'C2'])]
    for mode, col in (('external_only', 'reach_external'), ('universe', 'reach_universe')):
        s = recall_sweep(IDX, qf['neutral_mass'].to_numpy(float), qf['truth_id'].to_numpy(), PPM, reachable=qf[col].to_numpy(),
                         exclude_mask=excl, query_ids=qf['query_id'].to_numpy(), has_reference_mask=has_ref_f)
        SWEEPS.append(s.assign(mode=mode, fold=f))
    print(f'fold {f}: {len(qf):,} C1/C2 queries done')
SWEEP = pd.concat(SWEEPS, ignore_index=True).merge(
    Q[['query_id', 'regime', 'neutral_mass', 'ionization_mode', 'adduct', 'instrument_type', 'formula_class', 'truth_sources', 'true_connectivity_key']],
    on='query_id', how='left')

In [ ]:
from casmi.candidates.recall import recall_by_ppm, gate_a_assessment
K = CFG['recall']['k_values']
SUMMARY = recall_by_ppm(SWEEP, k_values=K, group_cols=('mode', 'regime'))
display(SUMMARY[['mode', 'regime', 'ppm', 'n_queries', 'recall_all', 'recall_at_25', 'recall_at_100', 'recall_at_500',
                 'pool_size_median', 'pool_size_p90', 'pool_size_p99']])
c2_ext = SUMMARY[(SUMMARY['mode'] == 'external_only') & (SUMMARY['regime'] == 'C2')]
GATE_A = gate_a_assessment(c2_ext, ppm_ref=10)
print(json.dumps(GATE_A, indent=2))
SHORTCUT = SWEEP[SWEEP['regime'] == 'C2'].groupby('ppm')['pool_share_with_reference'].describe()
display(SHORTCUT)

In [ ]:
# breakdowns of C2 external_only recall by mass / ion mode / adduct / formula class / truth source database
from casmi.validation.reporting import add_strata, stratified_table
from casmi.validation.metrics import candidate_stage_metrics
c2 = SWEEP[(SWEEP['mode'] == 'external_only') & (SWEEP['regime'] == 'C2')]
c2 = add_strata(c2.drop(columns=['neutral_mass', 'ionization_mode', 'adduct', 'instrument_type']), Q, CFG['validation']['mass_bins_da'],
                CFG['validation']['pool_size_bins'], group_cols=('fold', 'true_connectivity_key', 'ppm'))
BREAKDOWN = pd.concat([stratified_table(g, candidate_stage_metrics, strata=('mass_bin', 'ionization_mode', 'adduct', 'formula_class',
                                                                            'truth_sources', 'pool_size_bin'), k_values=K, min_queries=20).assign(ppm=ppm)
                       for ppm, g in c2.groupby('ppm')], ignore_index=True)
display(BREAKDOWN[(BREAKDOWN.ppm == 10) & (BREAKDOWN.regime == 'C2')].sort_values(['stratum', 'recall_all']).head(60))

In [ ]:
from casmi.viz import plots_v2
FIG = P.reports_dir / 'c2_candidate_recall_figures'; FIG.mkdir(parents=True, exist_ok=True)
s = SUMMARY[SUMMARY['regime'] == 'C2']
plots_v2.recall_vs_ppm(s[s['mode'] == 'external_only'], title='C2 recall vs ppm (external_only)').savefig(FIG / 'c2_recall_external.png', dpi=120)
plots_v2.recall_vs_ppm(s[s['mode'] == 'universe'], title='C2 recall vs ppm (universe)').savefig(FIG / 'c2_recall_universe.png', dpi=120)
plots_v2.pool_size_distribution(SWEEP[(SWEEP['regime'] == 'C2') & (SWEEP['mode'] == 'universe')]).savefig(FIG / 'c2_pool_sizes.png', dpi=120)

In [ ]:
from casmi.validation.reporting import save_report
from casmi.workspace.artifacts import write_metadata
from casmi.workspace.experiments import log_experiment
save_report(SUMMARY, {'notebook': NOTEBOOK, 'gate_a': GATE_A, 'regime_c2_mode': REG_META['c2_mode'], 'ppm_grid': PPM, 'k_values': K,
                      'n_c2_queries': int((Q['regime'] == 'C2').sum()),
                      'shortcut_pool_share_with_reference_median_by_ppm': SHORTCUT['50%'].to_dict(),
                      'universe_manifest': json.loads((UNIVERSE_ROOT / 'universe_manifest.json').read_text())},
            P.reports_dir, 'c2_candidate_recall')
BREAKDOWN.to_parquet(P.reports_dir / 'c2_candidate_recall_breakdown.parquet', index=False)
SWEEP.to_parquet(P.reports_dir / 'c2_candidate_recall_per_query.parquet', index=False)
write_metadata(FIG, NOTEBOOK, config={'recall': CFG['recall']}, seed=SEED, repo_root=REPO_ROOT,
               input_paths={'regimes': P.processed_dir / 'validation_regimes.parquet', 'universe': UNIVERSE_ROOT})
row = c2_ext[c2_ext['ppm'] == 10]
log_experiment(P.experiments_dir, 'gateA_c2_recall_10ppm', 'C2 external_only candidate recall at 10 ppm (mass order)', ['mass'],
               config_path=CFG['_config_path'], candidate_recall=float(row['recall_all'].iloc[0]) if len(row) else None,
               notes=GATE_A['decision'])
print('saved reports/c2_candidate_recall.parquet / .json')